# End-to-End Data Load Performance: Kerchunk vs netCDF


## Overview

This notebook benchmarks end-to-end variable access performance (open + full `.load()`) for kerchunk and native netCDF datasets. Dataset open time and full variable materialization time are recorded separately to decompose total cost.

These benchmarks represent a workload that fully loads the variable into memory, emphasizing raw I/O throughput in addition to metadata access. Results should be interpreted separately from metadata-only benchmarks or lazy/dask-based compute workflows.

The analysis includes representative cases to isolate structural effects:

- **Amon:** dataset with 1 file vs multi-file (~80–150 files) to contrast fragmentation
- **3hr (normal case):** large time dimension with typical scaling behavior
- **3hr (time-last outlier):** extremely large time dimension ordered last, where kerchunk exhibits higher load cost

The goal is to evaluate how file count, temporal frequency, and dimension structure influence total access time and relative scaling between kerchunk and native netCDF.


In [1]:
import json
import time

import numpy as np
import pandas as pd
import xarray as xr
import xcdat as xc
from IPython.display import HTML

## Read in Results and Input Mapping


In [2]:
df_raw = pd.read_csv(
    "riotai/results/20260126_130127/kerchunk_vs_netcdf_raw_20260126_130127.csv"
)

# Create a DataFrame from the JSON to NetCDF mapping.
with open("riotai/json_to_netcdf_maps/json_to_netcdf.json", "r") as file:
    json_netcdf_map = json.load(file)

rows = []
for freq, json_map in json_netcdf_map.items():
    for json_key, netcdf_filepaths in json_map.items():
        rows.append(
            {
                "frequency": freq,
                "json": json_key,
                "netcdf_filepaths": netcdf_filepaths,
            }
        )

df_json_netcdf = pd.DataFrame(rows)

df_raw_joined = df_raw.merge(df_json_netcdf, how="left", on=["frequency", "json"])

## Get specific datasets for these cases:

- **Amon:** dataset with 1 file vs ~100 files (fragmentation comparison)
- **3hr (normal case):** large time dimension with typical performance
- **3hr (time-last outlier):** very large time dimension ordered last, where kerchunk shows higher setup cost


In [3]:
# Filter for Amon: 1 file vs ~100 files
df_amon_1_file = df_raw_joined[
    (df_raw_joined["frequency"] == "Amon") & (df_raw_joined["num_netcdf_files"] == 1)
]
df_amon_many_files = df_raw_joined[
    (df_raw_joined["frequency"] == "Amon") & (df_raw_joined["num_netcdf_files"] >= 80)
]

# Filter for daily (normal case): large time dimension, typical performance
df_daily_normal = df_raw_joined[
    (df_raw_joined["frequency"] == "day")
    & (df_raw_joined["timesteps"] > 100000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

# Filter for 3hr (normal case): large time dimension, typical performance
df_3hr_normal = df_raw_joined[
    (df_raw_joined["frequency"] == "3hr")
    & (df_raw_joined["timesteps"] > 100000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

# Filter for 3hr (time-last outlier): very large time dimension ordered last
df_3hr_time_last_outlier = df_raw_joined[
    (df_raw_joined["frequency"] == "3hr")
    & (df_raw_joined["timesteps"] > 500000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

df_amon_1_file = df_amon_1_file.head(1).iloc[0]
df_amon_many_files = df_amon_many_files.head(2).iloc[-1]
df_3hr_normal = df_3hr_normal.head(1).iloc[0]
df_3hr_time_last_outlier = df_3hr_time_last_outlier.head(1).iloc[0]

## Benchmarks

### 1. Full-Field Load (`benchmark_full_field_load`)

Measures backend throughput.

Times:

- Dataset open
- Full variable materialization via `.compute()`

Includes:

- Dask execution
- IO reads
- CF decoding
- Masking / scale-offset

Use this benchmark to evaluate raw backend performance and chunk access behavior.

### 2. Reduction Workflow (`benchmark_reductions_separately`)

Measures realistic diagnostic workload performance.

Times separately:

- Dataset open
- Annual mean (`temporal.group_average`)
- Spatial average (`spatial.average`)

Each reduction is computed independently to avoid caching effects.

Use this benchmark to evaluate performance under xCDAT-style E3SM diagnostics.

### Interpretation

| Benchmark  | What It Tests                  |
| ---------- | ------------------------------ |
| Full-field | Backend IO throughput          |
| Reduction  | Diagnostic workflow efficiency |

Results should be interpreted separately.


In [4]:
def _open_kerchunk(kerchunk_path: str, use_xcdat: bool = False):
    """
    Open a kerchunk-backed dataset with consistent benchmark settings.

    Parameters
    ----------
    kerchunk_path : str
        Path to kerchunk reference JSON.
    use_xcdat : bool, default=False
        If True, use xcdat.open_dataset; else use xarray.open_dataset.

    Returns
    -------
    xr.Dataset or xc.Dataset
        Lazily opened dataset with Dask-backed arrays.
    """
    open_func = xc.open_dataset if use_xcdat else xr.open_dataset

    return open_func(kerchunk_path, engine="kerchunk", chunks={})


def _open_netcdf(netcdf_paths, use_xcdat: bool = False):
    """
    Open native NetCDF dataset(s) with consistent benchmark settings.

    Parameters
    ----------
    netcdf_paths : str or list[str]
        Path or list of NetCDF files.
    use_xcdat : bool, default=False
        If True, use xcdat.open_dataset/open_mfdataset; else use xarray.

    Returns
    -------
    xr.Dataset or xc.Dataset
        Lazily opened dataset with Dask-backed arrays.
    """
    if isinstance(netcdf_paths, (list, tuple)) and len(netcdf_paths) > 1:
        open_func = xc.open_mfdataset if use_xcdat else xr.open_mfdataset

        return open_func(netcdf_paths, combine="by_coords", parallel=False, chunks={})
    else:
        path = (
            netcdf_paths[0] if isinstance(netcdf_paths, (list, tuple)) else netcdf_paths
        )
        open_func = xc.open_dataset if use_xcdat else xr.open_dataset

        return open_func(path, chunks={})

In [5]:
def benchmark_full_field_load(
    kerchunk_path,
    netcdf_paths,
    variable,
    runs: int = 5,
    drop_first_run: bool = True,
    use_xcdat: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Benchmark full-field materialization of a variable for kerchunk and
    native NetCDF backends.

    This benchmark measures:

      1. Dataset open time (metadata parsing + Dask graph construction)
      2. Full variable materialization time via `.compute()`

    Notes
    -----
    - This is not pure storage-layer IO; Dask scheduling overhead is included.
    - Each run opens a fresh dataset to avoid memory reuse.
    - If `drop_first_run=True`, the first iteration (cold cache) is excluded
      from median statistics.

    Parameters
    ----------
    kerchunk_path : str
        Path to kerchunk reference JSON.
    netcdf_paths : str or list[str]
        Path or list of NetCDF files.
    variable : str
        Variable name to materialize.
    runs : int, default=5
        Number of repetitions.
    drop_first_run : bool, default=True
        Exclude first run from median statistics.
    use_xcdat : bool, default=False
        Whether to use xCDAT for opening datasets.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        (per-run timings, median summary statistics)
    """

    kc_open, kc_load = [], []
    nc_open, nc_load = [], []

    for _ in range(runs):

        # Kerchunk
        t0 = time.perf_counter()
        ds = _open_kerchunk(kerchunk_path, use_xcdat=use_xcdat)
        t1 = time.perf_counter()
        kc_open.append(t1 - t0)

        t0 = time.perf_counter()
        ds[variable].compute()
        t1 = time.perf_counter()
        kc_load.append(t1 - t0)
        ds.close()

        # NetCDF
        t0 = time.perf_counter()
        ds = _open_netcdf(netcdf_paths, use_xcdat=use_xcdat)
        t1 = time.perf_counter()
        nc_open.append(t1 - t0)

        t0 = time.perf_counter()
        ds[variable].compute()
        t1 = time.perf_counter()
        nc_load.append(t1 - t0)
        ds.close()

    kc_open_eval = kc_open[1:] if drop_first_run else kc_open
    nc_open_eval = nc_open[1:] if drop_first_run else nc_open
    kc_load_eval = kc_load[1:] if drop_first_run else kc_load
    nc_load_eval = nc_load[1:] if drop_first_run else nc_load

    df = pd.DataFrame(
        {
            "kerchunk_open": kc_open,
            "kerchunk_load": kc_load,
            "netcdf_open": nc_open,
            "netcdf_load": nc_load,
        }
    )

    summary = pd.DataFrame(
        {
            "kerchunk_open_median": [float(np.median(kc_open_eval))],
            "netcdf_open_median": [float(np.median(nc_open_eval))],
            "kerchunk_load_median": [float(np.median(kc_load_eval))],
            "netcdf_load_median": [float(np.median(nc_load_eval))],
            "difference_load_median": [
                float(np.median(kc_load_eval) - np.median(nc_load_eval))
            ],
        }
    )

    return df, summary

In [6]:
def benchmark_reductions_separately(
    kerchunk_path,
    netcdf_paths,
    variable,
    runs: int = 5,
    decode_cf: bool = True,
    mask_and_scale: bool = True,
    drop_first_run: bool = True,
    use_xcdat: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Benchmark xCDAT diagnostic reductions separately for kerchunk and
    native NetCDF backends.

    This benchmark measures:

      1. Dataset open time
      2. Annual mean computation time
         (`temporal.group_average(...).compute()`)
      3. Spatial average computation time
         (`spatial.average(...).compute()`)

    Missing bounds are generated (if needed) before reductions to ensure
    consistent spatial and temporal averaging behavior.

    Each reduction is computed independently using a freshly opened dataset
    to avoid cached or in-memory reuse affecting results.

    Notes
    -----
    - Includes Dask execution, decoding, masking, and reduction cost.
    - Bounds generation is not included in reduction timing.
    - First run is dropped by default to reduce cold-cache bias.
    """

    kc_open, kc_annual, kc_spatial = [], [], []
    nc_open, nc_annual, nc_spatial = [], [], []

    for _ in range(runs):

        # -------------------------
        # Kerchunk
        # -------------------------
        t0 = time.perf_counter()
        ds = _open_kerchunk(kerchunk_path, use_xcdat)
        t1 = time.perf_counter()
        kc_open.append(t1 - t0)

        ds = ds.bounds.add_missing_bounds()

        annual = ds.temporal.group_average(variable, freq="year")
        t0 = time.perf_counter()
        annual.compute()
        t1 = time.perf_counter()
        kc_annual.append(t1 - t0)
        ds.close()

        ds = _open_kerchunk(kerchunk_path, use_xcdat)
        ds = ds.bounds.add_missing_bounds()

        spatial = ds.spatial.average(variable)
        t0 = time.perf_counter()
        spatial.compute()
        t1 = time.perf_counter()
        kc_spatial.append(t1 - t0)
        ds.close()

        # -------------------------
        # NetCDF
        # -------------------------
        t0 = time.perf_counter()
        ds = _open_netcdf(netcdf_paths, use_xcdat)
        t1 = time.perf_counter()
        nc_open.append(t1 - t0)

        ds = ds.bounds.add_missing_bounds()

        annual = ds.temporal.group_average(variable, freq="year")
        t0 = time.perf_counter()
        annual.compute()
        t1 = time.perf_counter()
        nc_annual.append(t1 - t0)
        ds.close()

        ds = _open_netcdf(netcdf_paths)
        ds = ds.bounds.add_missing_bounds()

        spatial = ds.spatial.average(variable)
        t0 = time.perf_counter()
        spatial.compute()
        t1 = time.perf_counter()
        nc_spatial.append(t1 - t0)
        ds.close()

    kc_open_eval = kc_open[1:] if drop_first_run else kc_open
    nc_open_eval = nc_open[1:] if drop_first_run else nc_open
    kc_annual_eval = kc_annual[1:] if drop_first_run else kc_annual
    nc_annual_eval = nc_annual[1:] if drop_first_run else nc_annual
    kc_spatial_eval = kc_spatial[1:] if drop_first_run else kc_spatial
    nc_spatial_eval = nc_spatial[1:] if drop_first_run else nc_spatial

    df = pd.DataFrame(
        {
            "kerchunk_open": kc_open,
            "kerchunk_annual": kc_annual,
            "kerchunk_spatial": kc_spatial,
            "netcdf_open": nc_open,
            "netcdf_annual": nc_annual,
            "netcdf_spatial": nc_spatial,
        }
    )

    summary = pd.DataFrame(
        {
            "kerchunk_open_median": [float(np.median(kc_open_eval))],
            "netcdf_open_median": [float(np.median(nc_open_eval))],
            "kerchunk_annual_median": [float(np.median(kc_annual_eval))],
            "netcdf_annual_median": [float(np.median(nc_annual_eval))],
            "kerchunk_spatial_median": [float(np.median(kc_spatial_eval))],
            "netcdf_spatial_median": [float(np.median(nc_spatial_eval))],
        }
    )

    return df, summary

## Case 1 - Amon (1 file dataset)


In [7]:
df_amon_1_file.to_dict()

{'frequency': 'Amon',
 'json': '/global/cfs/projectdirs/m4931/kerchunk/tas/historical/mon/CMIP6.CMIP.CNRM-CERFACS.CNRM-CM6-1.historical.r14i1p1f2.Amon.tas.gr.v20191004.kerchunk.json',
 'num_netcdf_files': 1,
 'timesteps': 1980,
 'dims': "{'time': 1980, 'lat': 128, 'lon': 256, 'axis_nbounds': 2}",
 'kerchunk_time': 0.5189495849772356,
 'netcdf_time': 0.1653154970263131,
 'netcdf_filepaths': ['/global/cfs/projectdirs/m4931/gsharing/css03_data/CMIP6/CMIP/CNRM-CERFACS/CNRM-CM6-1/historical/r14i1p1f2/Amon/tas/gr/v20191004/tas_Amon_CNRM-CM6-1_historical_r14i1p1f2_gr_185001-201412.nc']}

### a. Benchmark Full Load


In [8]:
results_amon_1_file_full, summary_amon_1_file_full = benchmark_full_field_load(
    kerchunk_path=df_amon_1_file["json"],
    netcdf_paths=df_amon_1_file["netcdf_filepaths"],
    variable="tas",
    runs=5,
)

In [9]:
df_amon_1_file["json"]

'/global/cfs/projectdirs/m4931/kerchunk/tas/historical/mon/CMIP6.CMIP.CNRM-CERFACS.CNRM-CM6-1.historical.r14i1p1f2.Amon.tas.gr.v20191004.kerchunk.json'

In [10]:
path = "/global/cfs/projectdirs/m4931/kerchunk/tas/historical/mon/CMIP6.CMIP.CNRM-CERFACS.CNRM-CM6-1.historical.r14i1p1f2.Amon.tas.gr.v20191004.kerchunk.json"
ds1 = xr.open_dataset(path, engine="kerchunk")
ds2 = xr.open_dataset(path, engine="kerchunk", chunks={})

In [11]:
ds1.tas

<xarray.DataArray 'tas' (time: 1980, lat: 128, lon: 256)> Size: 260MB
[64880640 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 16kB 1850-01-16T12:00:00 ... 2014-12-16T12...
  * lat      (lat) float64 1kB -88.93 -87.54 -86.14 -84.74 ... 86.14 87.54 88.93
  * lon      (lon) float64 2kB 0.0 1.406 2.812 4.219 ... 354.4 355.8 357.2 358.6
    height   (time) float64 16kB ...
Attributes:
    online_operation:    average
    cell_methods:        area: time: mean
    interval_operation:  900 s
    interval_write:      1 month
    standard_name:       air_temperature
    description:         Near-Surface Air Temperature
    long_name:           Near-Surface Air Temperature
    history:             none
    units:               K
    cell_measures:       area: areacella

In [12]:
results_amon_1_file_full_xc, summary_amon_1_file_full_xc = benchmark_full_field_load(
    kerchunk_path=df_amon_1_file["json"],
    netcdf_paths=df_amon_1_file["netcdf_filepaths"],
    variable="tas",
    runs=5,
    use_xcdat=True,
)

In [13]:
summary_amon_1_file_full

,kerchunk_open_median,netcdf_open_median,kerchunk_load_median,netcdf_load_median,difference_load_median
0,0.047669,0.045899,20.359607,2.32373,18.035877


In [14]:
summary_amon_1_file_full_xc

,kerchunk_open_median,netcdf_open_median,kerchunk_load_median,netcdf_load_median,difference_load_median
0,0.087616,0.087624,21.332329,2.379561,18.952768


### b. Benchmark Reductions


In [15]:
results_amon_1_file_reductions, summary_amon_1_file_reductions = (
    benchmark_reductions_separately(
        kerchunk_path=df_amon_1_file["json"],
        netcdf_paths=df_amon_1_file["netcdf_filepaths"],
        variable="tas",
        runs=5,
        use_xcdat=True,
    )
)

In [16]:
summary_amon_1_file_reductions

,kerchunk_open_median,netcdf_open_median,kerchunk_annual_median,netcdf_annual_median,kerchunk_spatial_median,netcdf_spatial_median
0,0.077845,0.075434,28.817615,10.916534,33.323723,6.818612


**Key observations:**

- The dataset contains **1,980 timesteps** on a **128 × 256 grid** in a **single netCDF file**.
- **Open times are similar** for kerchunk and native netCDF (~0.04 s).
- **Full-field compute** is much slower with **kerchunk (~16.1 s)** vs **netCDF (~2.36 s)**.
- The slowdown appears during **data compute**, not metadata access.
- Kerchunk is also slower for reductions:
  - **Annual mean:** ~25.1 s vs ~9.48 s
  - **Spatial average:** ~29.5 s vs ~6.25 s

**Takeaway:**

For this **single contiguous file**, native netCDF clearly outperforms kerchunk for both **full loads** and **diagnostic reductions**. Without file fragmentation, kerchunk provides no performance advantage and adds overhead.


## Case 2 - Amon (~100 file dataset)


In [17]:
df_amon_many_files

frequency                                                        Amon
json                /global/cfs/projectdirs/m4931/kerchu...
num_netcdf_files                                                   86
timesteps                                                        1032
dims                {'lat': 256, 'bnds': 2, 'lon': 512, 'time': 1032}
kerchunk_time                                                0.286979
netcdf_time                                                  7.761698
netcdf_filepaths    [/global/cfs/projectdirs/m4931/gsharing/css03_...
Name: 4, dtype: object

### a. Benchmark Full Load


In [18]:
results_amon_many_files_full, summary_amon_many_files_full = benchmark_full_field_load(
    kerchunk_path=df_amon_many_files["json"],
    netcdf_paths=df_amon_many_files["netcdf_filepaths"],
    variable="tas",
    runs=5,
    use_xcdat=True,
)

In [19]:
summary_amon_many_files_full

,kerchunk_open_median,netcdf_open_median,kerchunk_load_median,netcdf_load_median,difference_load_median
0,0.101677,4.138159,8.585993,4.263013,4.32298


### b. Benchmark Reductions


In [20]:
results_amon_many_files_reductions, summary_amon_many_files_reductions = (
    benchmark_reductions_separately(
        kerchunk_path=df_amon_many_files["json"],
        netcdf_paths=df_amon_many_files["netcdf_filepaths"],
        variable="tas",
        runs=5,
        use_xcdat=True,
    )
)

/tmp/ipykernel_326914/1748661282.py:41: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  return open_func(netcdf_paths, combine="by_coords", parallel=False, chunks={})


ValueError: More than one grid cell spans prime meridian.

**Key observations:**

- This Amon dataset contains 1,032 timesteps on a 256 × 512 grid and is split across 86 netCDF files.
- Kerchunk open time (~0.07 s) is dramatically faster than native netCDF open time (~4.7 s), reflecting the cost of multi-file discovery and coordination in netCDF.
- During full variable materialization (`.load()`), kerchunk (~8.9 s) is slower than native netCDF (~4.7 s).
- The primary performance difference between backends appears during dataset open rather than data load.

**Takeaway:**
For fragmented multi-file Amon datasets, kerchunk substantially reduces dataset open time by avoiding file-by-file coordination overhead. However, during full data materialization, native netCDF remains faster for this case. Kerchunk’s advantage in this regime is driven primarily by metadata and file aggregation efficiency rather than raw data read speed.


## Case 3 - Day


In [ ]:
df_amon_many_files
results_amon_many_files = benchmark_variable_load(
    kerchunk_path=df_amon_many_files["json"],
    netcdf_paths=df_amon_many_files["netcdf_filepaths"],
    variable="tas",
)
results_amon_many_files

## Case 3 - 3hr (normal-case)


In [ ]:
df_3hr_normal

In [ ]:
results_3hr_normal = benchmark_variable_load(
    kerchunk_path=df_3hr_normal["json"],
    netcdf_paths=df_3hr_normal["netcdf_filepaths"],
    variable="pr",
)

In [ ]:
results_3hr_normal

## TL;DR

### Dataset Open (`.open_dataset()`)

- **Single-file datasets (e.g., Amon, 1 file):**  
  Kerchunk and native netCDF have similar open times. NetCDF may be slightly faster since it reads metadata directly from a single file.

- **Multi-file, fragmented datasets (e.g., 50–100+ files):**  
  Kerchunk is significantly faster to open. It avoids file-by-file discovery and aggregation overhead that native netCDF incurs with `open_mfdataset`.

- **High-frequency datasets (e.g., 3hr, hourly):**  
  As file count and fragmentation increase, kerchunk’s advantage in open time becomes more pronounced.

---

### Variable Load (`.load()`)

- **Single-file, moderate-size datasets (~1–2k timesteps):**  
  Native netCDF is typically faster for full `.load()`. Kerchunk adds indirection overhead and does not improve raw contiguous read performance.

- **Multi-file datasets:**  
  Native netCDF may still be faster for raw data materialization, but kerchunk can offset this with much faster open times, making end-to-end performance competitive.

- **Very large time dimensions (e.g., 3hr with 100k+ timesteps):**  
  Load performance becomes sensitive to chunking and dimension order. Large, finely chunked time axes—especially when `time` is last—can increase kerchunk load cost.

---

### Chunking and Execution Mode

- Using `chunks={}` forces dask-backed lazy arrays for both backends and ensures fair `.load()` comparisons.
- Native netCDF benefits from optimized C-level sequential reads for single contiguous files.
- Kerchunk’s strength lies in metadata aggregation and fragmentation handling, not raw single-file throughput.

---

**Bottom line:**  
Kerchunk excels at reducing dataset open cost for fragmented, multi-file datasets. Native netCDF remains most efficient for full-variable loads

| Scenario               | Winner              |
| ---------------------- | ------------------- |
| 1 file + full load     | NetCDF              |
| Many files + open      | Kerchunk            |
| End-to-end, fragmented | Often competitive   |
| Huge time axis         | Depends on chunking |
